<a href="https://colab.research.google.com/github/qyasbeb/smartshop-ai/blob/main/epidemio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

patients = pd.read_csv("patients.csv")
mesures = pd.read_csv("mesures_nettoyees.csv")

print(patients.shape)
print(patients.columns)
print(patients.dtypes)

print(mesures.shape)
print(mesures.columns)
print(mesures.dtypes)

(12, 4)
Index(['id_patient', 'age', 'service', 'sexe'], dtype='object')
id_patient    object
age            int64
service       object
sexe          object
dtype: object
(29, 6)
Index(['id_patient', 'date', 'temperature', 'frequence_cardiaque',
       'temperature_superieure_37_5', 'jour_semaine'],
      dtype='object')
id_patient                      object
date                            object
temperature                    float64
frequence_cardiaque            float64
temperature_superieure_37_5     object
jour_semaine                    object
dtype: object


In [4]:
identifiants_absents = set(mesures["id_patient"]) - set(patients["id_patient"])

print(identifiants_absents)
print(len(identifiants_absents))

{'P999'}
1


In [6]:
mesures_patients = pd.merge(
    mesures,
    patients,
    on="id_patient",
    how="inner"
)

print(mesures_patients)

   id_patient        date  temperature  frequence_cardiaque  \
0        P001  2026-01-05         36.8                 72.0   
1        P001  2026-01-06         37.1                 78.0   
2        P001  2026-01-07         36.9                 74.0   
3        P002  2026-01-05         37.5                 84.0   
4        P002  2026-01-06         36.9                 76.0   
5        P002  2026-01-08          NaN                 81.0   
6        P003  2026-01-05         38.2                 96.0   
7        P003  2026-01-06         37.8                 91.0   
8        P003  2026-01-09         37.0                 80.0   
9        P004  2026-01-06         36.7                 68.0   
10       P004  2026-01-07         36.8                 70.0   
11       P005  2026-01-05         37.4                 88.0   
12       P005  2026-01-08         37.1                  NaN   
13       P005  2026-01-09         36.9                 79.0   
14       P006  2026-01-05         36.6                 

In [7]:
print("Avant :", len(mesures))
print("Après :", len(mesures_patients))

Avant : 29
Après : 28


In [8]:
mesures_gauche = pd.merge(
    mesures,
    patients,
    on="id_patient",
    how="left"
)

lignes_absentes = mesures_gauche[
    mesures_gauche["age"].isna()
]

print(lignes_absentes)

   id_patient        date  temperature  frequence_cardiaque  \
28       P999  2026-01-10         37.0                 75.0   

   temperature_superieure_37_5 jour_semaine  age service sexe  
28                       False     Saturday  NaN     NaN  NaN  


In [9]:
synthese = mesures_patients.groupby("service").agg(
    nombre_mesures=("id_patient", "count"),
    nombre_patients=("id_patient", "nunique"),
    temperature_moyenne=("temperature", "mean")
)

print(synthese)

                   nombre_mesures  nombre_patients  temperature_moyenne
service                                                                
Cardiologie                    10                4            37.160000
Medecine generale               9                4            37.012500
Pneumologie                     9                4            37.655556


In [10]:
synthese = synthese.sort_values(
    by="temperature_moyenne",
    ascending=False
)

print(synthese)

                   nombre_mesures  nombre_patients  temperature_moyenne
service                                                                
Pneumologie                     9                4            37.655556
Cardiologie                    10                4            37.160000
Medecine generale               9                4            37.012500


In [11]:
synthese = synthese.reset_index()

synthese.columns = [
    "service",
    "nombre_de_mesures",
    "nombre_de_patients_distincts",
    "temperature_moyenne"
]

print(synthese)

             service  nombre_de_mesures  nombre_de_patients_distincts  \
0        Pneumologie                  9                             4   
1        Cardiologie                 10                             4   
2  Medecine generale                  9                             4   

   temperature_moyenne  
0            37.655556  
1            37.160000  
2            37.012500  


In [12]:
synthese.to_csv("service_synthese.csv", index=False)

In [13]:
nombre_services = mesures_patients["service"].nunique()
nombre_lignes = len(synthese)

print(nombre_services)
print(nombre_lignes)
#Le nombre de lignes correspond bien au nombre de services représentés. La vérification est donc correcte.

3
3
